In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from datetime import datetime
CARPETA_DATOS = Path("datos_dia_13")
CARPETA_DATOS.mkdir(exist_ok=True)

RUTA_DB = CARPETA_DATOS / "ventas_dia_13.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())


Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_13

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_13\ventas_dia_13.db


In [2]:
conexion = sqlite3.connect(RUTA_DB)

print("Conexión creada correctamente.")

conexion.close()

print("Conexión cerrada correctamente.")

def obtener_conexion():
    """
    Crea una conexión con la base de datos SQLite.

    Buenas prácticas:
    - Activa la validación de llaves foráneas.
    - Permite acceder a las columnas por nombre.
    """
    conexion = sqlite3.connect(RUTA_DB)

    # En SQLite, las llaves foráneas deben activarse en cada conexión.
    conexion.execute("PRAGMA foreign_keys = ON")

    # Permite leer resultados como si fueran diccionarios.
    conexion.row_factory = sqlite3.Row

    return conexion

Conexión creada correctamente.
Conexión cerrada correctamente.


In [44]:
conexion = obtener_conexion()

estado_fk = pd.read_sql_query("""
PRAGMA foreign_keys
""", conexion)

conexion.close()

estado_fk

,foreign_keys
0,1


In [3]:
conexion = obtener_conexion()
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS prueba_row_factory")

cursor.execute("""
CREATE TABLE prueba_row_factory (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    edad INTEGER NOT NULL
)
""")

cursor.execute("""
INSERT INTO prueba_row_factory (id, nombre, edad)
VALUES (?, ?, ?)
""", (1, "Ana", 22))

conexion.commit()

cursor.execute("""
SELECT id, nombre, edad
FROM prueba_row_factory
""")

fila = cursor.fetchone()

conexion.close()

fila

In [8]:
print("ID:", fila["id"])
print("Nombre:", fila["nombre"])
print("Edad:", fila["edad"])

conexion.row_factory = sqlite3.Row

def consultar_df(sql, parametros=()):
    """
    Ejecuta una consulta SELECT y devuelve el resultado como DataFrame.

    sql:
        Consulta SQL con placeholders ?.

    parametros:
        Tupla con los valores que sustituyen los ?.
    """
    conexion = obtener_conexion()

    try:
        resultado = pd.read_sql_query(sql, conexion, params=parametros)
        return resultado

    finally:
        conexion.close()

ID: 1
Nombre: Ana
Edad: 22


In [7]:
def consultar_uno(sql, parametros=()):
    """
    Ejecuta una consulta SELECT y devuelve una sola fila.

    Si no hay resultados, devuelve None.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        fila = cursor.fetchone()
        return fila

    finally:
        conexion.close()

In [9]:
def consultar_varios(sql, parametros=()):
    """
    Ejecuta una consulta SELECT y devuelve varias filas.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        filas = cursor.fetchall()
        return filas

    finally:
        conexion.close()

In [10]:
def ejecutar(sql, parametros=()):
    """
    Ejecuta INSERT, UPDATE o DELETE.

    Devuelve:
    - ultimo_id: último ID insertado.
    - filas_afectadas: número de filas modificadas.

    Si ocurre un error, hace rollback.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        conexion.commit()

        resultado = {
            "ultimo_id": cursor.lastrowid,
            "filas_afectadas": cursor.rowcount
        }

        return resultado

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()

In [11]:
def ejecutar_varios(sql, lista_parametros):
    """
    Ejecuta una consulta varias veces usando executemany.

    Es útil para insertar datos iniciales.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.executemany(sql, lista_parametros)
        conexion.commit()

        return {
            "filas_afectadas": cursor.rowcount
        }

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()

In [12]:
def ejecutar_script(sql_script):
    """
    Ejecuta un bloque grande de SQL.

    Es útil para crear varias tablas.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.executescript(sql_script)
        conexion.commit()

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()

In [13]:
SQL_CREAR_TABLAS = """
DROP TABLE IF EXISTS detalle_ventas;
DROP TABLE IF EXISTS ventas;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    ciudad TEXT DEFAULT 'No especificada'
);

CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
);

CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    fecha TEXT NOT NULL,
    total REAL NOT NULL DEFAULT 0 CHECK(total >= 0),

    FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

CREATE TABLE detalle_ventas (
    id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0),
    subtotal REAL NOT NULL CHECK(subtotal >= 0),

    FOREIGN KEY (id_venta)
        REFERENCES ventas(id_venta)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (id_producto)
        REFERENCES productos(id_producto)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""

In [14]:
ejecutar_script(SQL_CREAR_TABLAS)

print("Tablas creadas correctamente.")

Tablas creadas correctamente.


In [15]:
tablas = consultar_df("""
SELECT name AS tabla
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

tablas

,tabla
0,clientes
1,detalle_ventas
2,productos
3,prueba_row_factory
4,sqlite_sequence
5,ventas


In [16]:
for tabla in ["clientes", "productos", "ventas", "detalle_ventas"]:
    print(f"Estructura de la tabla: {tabla}")
    display(consultar_df(f"PRAGMA table_info({tabla})"))

Estructura de la tabla: clientes


,cid,name,type,notnull,dflt_value,pk
0,0,id_cliente,INTEGER,0,NaN,1
1,1,nombre,TEXT,1,NaN,0
2,2,correo,TEXT,1,NaN,0
3,3,telefono,TEXT,0,NaN,0
4,4,ciudad,TEXT,0,'No especificada',0


Estructura de la tabla: productos


,cid,name,type,notnull,dflt_value,pk
0,0,id_producto,INTEGER,0,NaN,1
1,1,sku,TEXT,1,NaN,0
2,2,nombre,TEXT,1,NaN,0
3,3,categoria,TEXT,1,NaN,0
4,4,precio,REAL,1,NaN,0
5,5,stock,INTEGER,1,0,0
6,6,activo,INTEGER,1,1,0


Estructura de la tabla: ventas


,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,NaN,1
1,1,id_cliente,INTEGER,1,NaN,0
2,2,fecha,TEXT,1,NaN,0
3,3,total,REAL,1,0,0


Estructura de la tabla: detalle_ventas


,cid,name,type,notnull,dflt_value,pk
0,0,id_detalle,INTEGER,0,None,1
1,1,id_venta,INTEGER,1,None,0
2,2,id_producto,INTEGER,1,None,0
3,3,cantidad,INTEGER,1,None,0
4,4,precio_unitario,REAL,1,None,0
5,5,subtotal,REAL,1,None,0


In [17]:
for tabla in ["ventas", "detalle_ventas"]:
    print(f"Llaves foráneas de la tabla: {tabla}")
    display(consultar_df(f"PRAGMA foreign_key_list({tabla})")) 

Llaves foráneas de la tabla: ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,clientes,id_cliente,id_cliente,CASCADE,RESTRICT,NONE


Llaves foráneas de la tabla: detalle_ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,productos,id_producto,id_producto,CASCADE,RESTRICT,NONE
1,1,0,ventas,id_venta,id_venta,CASCADE,RESTRICT,NONE


In [18]:
CLIENTES_INICIALES = [
    ("Ana López", "ana@example.com", "555-111-2222", "CDMX"),
    ("Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara"),
    ("María Torres", "maria@example.com", None, "Monterrey"),
    ("Luis Romero", "luis@example.com", "555-888-9999", "Puebla")
]

PRODUCTOS_INICIALES = [
    ("P001", "Mouse inalámbrico", "Accesorios", 249.90, 15, 1),
    ("P002", "Teclado mecánico", "Accesorios", 899.00, 8, 1),
    ("P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 4, 1),
    ("P004", "Cable HDMI", "Cables", 120.00, 20, 1),
    ("P005", "Memoria USB 64GB", "Almacenamiento", 150.00, 30, 1),
    ("P006", "Bocinas Bluetooth", "Audio", 699.00, 10, 1),
    ("P007", "Producto sin ventas", "General", 100.00, 5, 1)
]

In [19]:
resultado_clientes = ejecutar_varios("""
INSERT INTO clientes (nombre, correo, telefono, ciudad)
VALUES (?, ?, ?, ?)
""", CLIENTES_INICIALES)

resultado_clientes

{'filas_afectadas': 4}

In [22]:
resultado_productos = ejecutar_varios("""
INSERT INTO productos (sku, nombre, categoria, precio, stock, activo)
VALUES (?, ?, ?, ?, ?, ?)
""", PRODUCTOS_INICIALES)

resultado_productos

{'filas_afectadas': 7}

In [20]:
clientes = consultar_df("""
SELECT *
FROM clientes
ORDER BY id_cliente
""")

clientes

,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey
3,4,Luis Romero,luis@example.com,555-888-9999,Puebla


In [23]:
productos = consultar_df("""
SELECT *
FROM productos
ORDER BY id_producto
""")

productos

,id_producto,sku,nombre,categoria,precio,stock,activo
0,1,P001,Mouse inalámbrico,Accesorios,249.9,15,1
1,2,P002,Teclado mecánico,Accesorios,899.0,8,1
2,3,P003,Monitor 24 pulgadas,Pantallas,3299.0,4,1
3,4,P004,Cable HDMI,Cables,120.0,20,1
4,5,P005,Memoria USB 64GB,Almacenamiento,150.0,30,1
5,6,P006,Bocinas Bluetooth,Audio,699.0,10,1
6,7,P007,Producto sin ventas,General,100.0,5,1


In [24]:
sku_usuario = "P001"

sql_mala_practica = f"""
SELECT *
FROM productos
WHERE sku = '{sku_usuario}'
"""

print(sql_mala_practica)


SELECT *
FROM productos
WHERE sku = 'P001'



In [25]:
sku_usuario = "P001"

producto = consultar_df("""
SELECT
    id_producto,
    sku,
    nombre,
    categoria,
    precio,
    stock
FROM productos
WHERE sku = ?
""", (sku_usuario,))

producto

,id_producto,sku,nombre,categoria,precio,stock
0,1,P001,Mouse inalámbrico,Accesorios,249.9,15


In [26]:
def buscar_producto_por_sku(sku):
    """
    Busca un producto activo por SKU.
    """
    producto = consultar_uno("""
    SELECT
        id_producto,
        sku,
        nombre,
        categoria,
        precio,
        stock,
        activo
    FROM productos
    WHERE sku = ?
    """, (sku,))

    return producto

In [46]:
producto = buscar_producto_por_sku("P003")

if producto is None:
    print("Producto no encontrado.")
else:
    print("Producto encontrado:")
    print("ID:", producto["id_producto"])
    print("SKU:", producto["sku"])
    print("Nombre:", producto["nombre"])
    print("Precio:", producto["precio"])
    print("Stock:", producto["stock"])

Producto encontrado:
ID: 3
SKU: P003
Nombre: Monitor 24 pulgadas
Precio: 3299.0
Stock: 4


In [28]:
def buscar_cliente_por_correo(correo):
    """
    Busca un cliente por su correo.
    """
    cliente = consultar_uno("""
    SELECT
        id_cliente,
        nombre,
        correo,
        telefono,
        ciudad
    FROM clientes
    WHERE correo = ?
    """, (correo,))

    return cliente

In [29]:
def registrar_cliente(nombre, correo, telefono=None, ciudad="No especificada"):
    """
    Registra un cliente nuevo.

    Usa consulta parametrizada y controla errores de integridad.
    """
    try:
        resultado = ejecutar("""
        INSERT INTO clientes (nombre, correo, telefono, ciudad)
        VALUES (?, ?, ?, ?)
        """, (nombre, correo, telefono, ciudad))

        print("Cliente registrado correctamente.")
        print("ID:", resultado["ultimo_id"])

        return resultado["ultimo_id"]

    except sqlite3.IntegrityError as error:
        print("No se pudo registrar el cliente.")
        print("Motivo:", error)
        return None

In [48]:
registrar_cliente(
    nombre="Filiberto",
    correo="fili@example.com",
    telefono=None,
    ciudad="CDmx"
)

Cliente registrado correctamente.
ID: 6


6

In [31]:
consultar_df("""
SELECT *
FROM clientes
ORDER BY id_cliente
""")

,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey
3,4,Luis Romero,luis@example.com,555-888-9999,Puebla
4,5,Sofía Herrera,sofia@example.com,NaN,Querétaro


In [32]:
registrar_cliente(
    nombre="Otra Ana",
    correo="ana@example.com",
    telefono=None,
    ciudad="CDMX"
)

No se pudo registrar el cliente.
Motivo: UNIQUE constraint failed: clientes.correo


In [33]:
def actualizar_stock_por_sku(sku, nuevo_stock):
    """
    Actualiza el stock de un producto usando SKU.

    Valida que el nuevo stock no sea negativo.
    """
    if nuevo_stock < 0:
        print("El stock no puede ser negativo.")
        return False

    resultado = ejecutar("""
    UPDATE productos
    SET stock = ?
    WHERE sku = ?
    """, (nuevo_stock, sku))

    if resultado["filas_afectadas"] == 0:
        print("No existe un producto con ese SKU.")
        return False

    print("Stock actualizado correctamente.")
    return True

In [51]:
actualizar_stock_por_sku("P001", 0)
consultar_df("""
SELECT sku, nombre, stock
FROM productos
WHERE sku = ?
""", ("P001",))


Stock actualizado correctamente.


,sku,nombre,stock
0,P001,Mouse inalámbrico,0


In [38]:
actualizar_stock_por_sku("NO_EXISTE", 10)

No existe un producto con ese SKU.


False

In [39]:
def desactivar_producto(sku):
    """
    Desactiva un producto sin borrarlo físicamente.

    Esta práctica conserva el historial de ventas.
    """
    resultado = ejecutar("""
    UPDATE productos
    SET activo = 0
    WHERE sku = ?
    """, (sku,))

    if resultado["filas_afectadas"] == 0:
        print("No existe un producto con ese SKU.")
        return False

    print("Producto desactivado correctamente.")
    return True

In [52]:
desactivar_producto("P089")

No existe un producto con ese SKU.


False

In [41]:
consultar_df("""
SELECT sku, nombre, activo
FROM productos
ORDER BY id_producto
""")

,sku,nombre,activo
0,P001,Mouse inalámbrico,1
1,P002,Teclado mecánico,1
2,P003,Monitor 24 pulgadas,1
3,P004,Cable HDMI,1
4,P005,Memoria USB 64GB,1
5,P006,Bocinas Bluetooth,1
6,P007,Producto sin ventas,0


In [42]:
def registrar_venta(id_cliente, productos_vendidos):
    """
    Registra una venta completa.

    productos_vendidos debe ser una lista de diccionarios con:
    - id_producto
    - cantidad

    Ejemplo:
    [
        {"id_producto": 1, "cantidad": 2},
        {"id_producto": 2, "cantidad": 1}
    ]

    Buenas prácticas:
    - Usa transacción.
    - Valida cliente.
    - Valida producto.
    - Valida stock.
    - Inserta encabezado.
    - Inserta detalle.
    - Actualiza stock.
    - Actualiza total.
    - Usa rollback si algo falla.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute("BEGIN")

        cursor.execute("""
        SELECT id_cliente
        FROM clientes
        WHERE id_cliente = ?
        """, (id_cliente,))

        cliente = cursor.fetchone()

        if cliente is None:
            raise ValueError("No existe el cliente indicado.")

        fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        cursor.execute("""
        INSERT INTO ventas (id_cliente, fecha, total)
        VALUES (?, ?, ?)
        """, (id_cliente, fecha, 0))

        id_venta = cursor.lastrowid
        total_venta = 0

        for item in productos_vendidos:
            id_producto = item["id_producto"]
            cantidad = item["cantidad"]

            if cantidad <= 0:
                raise ValueError("La cantidad debe ser mayor a cero.")

            cursor.execute("""
            SELECT
                id_producto,
                precio,
                stock,
                activo
            FROM productos
            WHERE id_producto = ?
            """, (id_producto,))

            producto = cursor.fetchone()

            if producto is None:
                raise ValueError(f"No existe el producto con ID {id_producto}.")

            if producto["activo"] != 1:
                raise ValueError(f"El producto con ID {id_producto} no está activo.")

            if producto["stock"] < cantidad:
                raise ValueError(
                    f"Stock insuficiente para producto ID {id_producto}. "
                    f"Disponible: {producto['stock']}, solicitado: {cantidad}."
                )

            precio_unitario = producto["precio"]
            subtotal = precio_unitario * cantidad
            total_venta += subtotal

            cursor.execute("""
            INSERT INTO detalle_ventas (
                id_venta,
                id_producto,
                cantidad,
                precio_unitario,
                subtotal
            )
            VALUES (?, ?, ?, ?, ?)
            """, (id_venta, id_producto, cantidad, precio_unitario, subtotal))

            cursor.execute("""
            UPDATE productos
            SET stock = stock - ?
            WHERE id_producto = ?
              AND stock >= ?
              AND activo = 1
            """, (cantidad, id_producto, cantidad))

            if cursor.rowcount == 0:
                raise ValueError(f"No se pudo actualizar el stock del producto ID {id_producto}.")

        cursor.execute("""
        UPDATE ventas
        SET total = ?
        WHERE id_venta = ?
        """, (total_venta, id_venta))

        conexion.commit()

        print("Venta registrada correctamente.")
        print("ID venta:", id_venta)
        print("Total:", total_venta)

        return id_venta

    except Exception as error:
        conexion.rollback()
        print("No se pudo registrar la venta.")
        print("Se deshicieron los cambios.")
        print("Motivo:", error)
        return None

    finally:
        conexion.close()

In [54]:
id_venta = registrar_venta(
    id_cliente=1,
    productos_vendidos=[
        {"id_producto": 1, "cantidad": 0},
        {"id_producto": 2, "cantidad": 1},
        {"id_producto": 3, "cantidad": 0}
    ]
)

id_venta

No se pudo registrar la venta.
Se deshicieron los cambios.
Motivo: La cantidad debe ser mayor a cero.
